<a href="https://colab.research.google.com/github/CaioAntonine/CaioAntonine_MVP_CEP/blob/main/MVP_CEP_2026/01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVP — CEP: Previsão de Falhas em Semicondutores

**Universidade de Brasília — Departamento de Engenharia de Produção**  
**Disciplina:** Ciência e Engenharia de Produção  
**Aluno:** Caio Antonine Pereira Gomes — 231013313  
**Professor:** Dr. Andre Luiz Marques Serrano  
**Data:** Abril de 2026

---

## Resumo

Este notebook apresenta o desenvolvimento de um MVP (Minimum Viable Product) para classificação supervisionada de qualidade em processos de fabricação de semicondutores. O objetivo é prever, a partir de leituras de sensores, se um chip será **aprovado (-1)** ou **reprovado (+1)** no teste final de qualidade, permitindo intervenção preventiva na linha de produção.

O dataset utilizado é o **SECOM** (Semiconductor Manufacturing Process), disponível publicamente no UCI Machine Learning Repository.

---
# 1. Definição do Problema

## 1.1 Contexto

A fabricação de semicondutores é um dos processos industriais mais complexos do mundo. Uma única linha de produção pode envolver centenas de etapas, cada uma monitorada por sensores que registram temperatura, pressão, fluxo de gases, tempo de exposição e dezenas de outras variáveis.

Defeitos microscópicos em qualquer etapa podem inutilizar um chip completamente. Identificar esses defeitos **somente ao final do processo** representa enorme desperdício de tempo e recursos. Um modelo capaz de antecipar falhas com base nas leituras dos sensores pode gerar economia significativa.

## 1.2 Objetivo

Construir um modelo de **classificação binária supervisionada** capaz de prever, a partir das leituras dos sensores de processo, se um chip semicondutor será **aprovado ou reprovado** no teste final de qualidade.

## 1.3 Pergunta Central

> Com base nas medições coletadas pelos sensores ao longo do processo de fabricação, é possível antecipar se um chip será reprovado no teste final, de forma a viabilizar intervenção preventiva e reduzir o custo de falhas tardias?

## 1.4 Dataset — SECOM (UCI Machine Learning Repository)

| Atributo | Detalhe |
|---|---|
| Instâncias | 1.567 registros de chips produzidos |
| Features | 590 atributos numéricos de sensores |
| Variável-alvo | Binária: +1 (reprovado) / -1 (aprovado) |
| Tipo de problema | Classificação binária |
| Dados faltantes | Sim — presença significativa |
| Link | https://archive.ics.uci.edu/dataset/179/secom |

## 1.5 Premissas e Hipóteses

**Premissa 1:** Os sensores capturam informação relevante sobre a qualidade final do chip, mesmo que individualmente nenhum sensor seja determinante.

**Premissa 2:** Os valores ausentes refletem falhas de leitura de sensores específicos e não são informativos por si só — serão tratados por imputação ou remoção.

**Hipótese 1:** A maioria das 590 features possui baixa variância ou alta correlação, tornando a seleção de atributos essencial.

## 1.6 Restrições do Projeto

- Apenas dados públicos do UCI serão utilizados
- Solução implementada integralmente em Python com bibliotecas open-source
- Notebook executável no Google Colab sem configuração adicional
- Divisão treino/teste será estratificada pela variável-alvo
- O escopo limita-se à predição de aprovação/reprovação; identificação da etapa causadora do defeito está fora do escopo

---
# 2. Instalação e Importação de Bibliotecas

In [3]:
!pip install imbalanced-learn --quiet

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Pré-processamento
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA

# Divisão dos dados
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# Modelos
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Desbalanceamento
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Avaliação
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_score,
    recall_score, ConfusionMatrixDisplay, RocCurveDisplay
)

# Reprodutibilidade
SEED = 7
np.random.seed(SEED)

---
# 3. Carga e Preparação dos Dados

## 3.1 Carregamento

O dataset SECOM é composto por dois arquivos:
- `secom.data`: matriz de features (1567 × 590)
- `secom_labels.data`: variável-alvo e timestamp

Vamos baixá-los diretamente do UCI.

In [6]:
# URLs do dataset no UCI
url_data   = "https://archive.ics.uci.edu/ml/machine-learning-databases/secom/secom.data"
url_labels = "https://archive.ics.uci.edu/ml/machine-learning-databases/secom/secom_labels.data"

# Carregamento
X_raw = pd.read_csv(url_data, sep=' ', header=None)
labels = pd.read_csv(url_labels, sep=' ', header=None)

y = labels[0]  # coluna 0 = resultado (-1 aprovado, +1 reprovado)

print(f"Shape dos dados: {X_raw.shape}")
print(f"Shape dos rótulos: {y.shape}")
print(f"\nDistribuição das classes:")
print(y.value_counts())
print(f"\nProporção de reprovados: {(y == 1).mean():.2%}")

Shape dos dados: (1567, 590)
Shape dos rótulos: (1567,)

Distribuição das classes:
0
-1    1463
 1     104
Name: count, dtype: int64

Proporção de reprovados: 6.64%


## 3.2 Análise Exploratória Inicial

In [7]:
# Visão geral dos dados
print("=== Primeiras linhas ===")
display(X_raw.head(3))

print(f"\n=== Dados Faltantes ===")
missing = X_raw.isnull().sum()
missing_pct = (missing / len(X_raw)) * 100
missing_df = pd.DataFrame({'Faltantes': missing, 'Percentual (%)': missing_pct})
print(f"Features sem nenhum NaN: {(missing == 0).sum()}")
print(f"Features com > 50% NaN: {(missing_pct > 50).sum()}")
print(f"Features com > 90% NaN: {(missing_pct > 90).sum()}")

=== Primeiras linhas ===


,0,1,2,3,4,5,6,7,8,9,...,580,581,582,583,584,585,586,587,588,589
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602



=== Dados Faltantes ===
Features sem nenhum NaN: 52
Features com > 50% NaN: 28
Features com > 90% NaN: 4
